# ArrayExpress from BioServices

[ArrayExpress](https://www.ebi.ac.uk/biostudies/arrayexpress) is a database of
functional genomics experiments hosted at EMBL-EBI. It contains gene expression
data from microarray and high-throughput sequencing studies.

ArrayExpress data is now part of the
[BioStudies](https://www.ebi.ac.uk/biostudies/) platform. BioServices wraps the
BioStudies REST API to give convenient Python access to ArrayExpress experiments.

In [1]:
from bioservices import ArrayExpress

s = ArrayExpress()

## Searching for experiments

Use `search()` to find experiments matching a free-text query. The result is a
dict containing pagination metadata and a list of hits.

In [2]:
res = s.search("breast cancer", page_size=5)
print("Total experiments found:", res["totalHits"])
print("Page:", res["page"], "/ page size:", res["pageSize"])

Total experiments found: 2525
Page: 1 / page size: 5


In [3]:
# Each hit contains accession, title, author, release date and file count
for hit in res["hits"]:
    print(hit["accession"], "|", hit["release_date"], "|", hit["title"][:60])

E-GEOD-17155 | 2010-05-16 | MicroRNA expression profiling of male breast cancer
E-GEOD-23891 | 2010-09-01 | Gene copy number variation in male breast cancer by aCGH
E-GEOD-33445 | 2011-11-11 | DNA hypermethylation in breast cancer: breast cancer tissues
E-GEOD-33447 | 2011-11-11 | Gene expression profiles in breast cancer: breast cancer tis
E-GEOD-33450 | 2011-11-11 | DNA hypermethylation and gene expression in breast cancer: b


### Sorting results

Results can be sorted by `relevance` (default), `release_date`, or `views`.

In [4]:
res_sorted = s.search("Homo sapiens RNA-seq", sort_by="release_date", sort_order="descending", page_size=5)
for hit in res_sorted["hits"]:
    print(hit["accession"], hit["release_date"])

E-MTAB-14552 2026-03-16
E-MTAB-15660 2026-03-16
E-MTAB-16743 2026-03-16
E-MTAB-16628 2026-03-16
E-MTAB-16735 2026-03-15


### Pagination

Use `page` and `page_size` to navigate large result sets.

In [5]:
# Retrieve the second page of 10 results
res_p2 = s.search("cancer", page=2, page_size=10)
print("Page:", res_p2["page"], "Results on this page:", len(res_p2["hits"]))

Page: 2 Results on this page: 10


## Retrieving a list of accessions

`queryAE()` is a convenience wrapper that returns only the accession strings.

In [6]:
accessions = s.queryAE("pneumonia homo sapiens", page_size=10)
print(accessions)

['E-MTAB-14588', 'E-GEOD-30385', 'E-GEOD-73189', 'E-MTAB-5638', 'E-MTAB-14564', 'E-MTAB-5815', 'E-GEOD-40012', 'E-MEXP-3001', 'E-GEOD-24338', 'E-GEOD-19314']


## Retrieving study metadata

`get_study()` returns the full metadata record for a specific experiment,
including title, organism, study type, protocols, and file listings.

In [7]:
study = s.get_study("E-MEXP-31")
print("Accession:", study["accno"])

# Extract top-level attributes
for attr in study["attributes"]:
    print(f"  {attr['name']:15s}: {attr['value'][:80]}")

Accession: E-MEXP-31
  Title          : Transcription profiling of mammalian male germ cells undergoing mitotic growth, 
  ReleaseDate    : 2004-03-01
  RootPath       : E-MEXP-31
  AttachTo       : ArrayExpress


In [8]:
# Extract organism and study type from the study section
for attr in study["section"]["attributes"]:
    print(f"  {attr['name']:20s}: {attr['value'][:80]}")

  Title               : Transcription profiling of mammalian male germ cells undergoing mitotic growth, 
  Study type          : transcription profiling by array
  Organism            : Rattus norvegicus
  Description         : We report a comprehensive large-scale expression profiling analysis of mammalian


## Listing files for an experiment

`get_files()` returns the names of all files associated with the experiment.

In [9]:
files = s.get_files("E-MEXP-31")
print(f"{len(files)} files in E-MEXP-31:")
# Show the main experiment description files (IDF/SDRF)
main_files = [f for f in files if f.endswith(".idf.txt") or f.endswith(".sdrf.txt")]
print("Main files:", main_files)

22 files in E-MEXP-31:
Main files: ['E-MEXP-31.idf.txt', 'E-MEXP-31.sdrf.txt']


## Downloading a file

`retrieve_file()` downloads a specific file and returns its content as a string
(plain text) or bytes (binary). Pass `save=True` to write it to disk.

In [10]:
idf = s.retrieve_file("E-MEXP-31", "E-MEXP-31.idf.txt")
# Print first 5 lines of the Investigation Description Format file
for line in idf.splitlines()[:5]:
    print(line)

Investigation Title	Transcription profiling of mammalian male germ cells undergoing mitotic growth, meiosis and gametogenesis in highly enriched cell populations
Comment[Submitted Name]	Rat Spermatogenesis
Experimental Design	development_or_differentiation_design	cell_type_comparison_design	transcription profiling by array
Experimental Design Term Source REF	mo:1.3.1.1	mo	EFO
Comment[ArrayExpressReleaseDate]	2004-03-01


## Legacy / backward-compatible methods

The original keyword-argument style is still supported for existing code.
Parameters such as `keywords`, `species`, `array`, `expdesign`, `exptype`,
`sortby` and `sortorder` are accepted and converted to the new API internally.

In [12]:
# Old-style search
res2 = s.queryExperiments(keywords="breast cancer", species="Homo sapiens")
print("Total hits (legacy call):", res2["totalHits"])

Total hits (legacy call): 1655


In [13]:
# Old-style file listing and download
files2 = s.retrieveFilesFromExperiment("E-MEXP-31")
print(files2[:4])

['SG2_u34a.CEL', 'SC1_u34a.CEL', 'ST2_u34b.CEL', 'SE1_u34b.CEL']
